# Week 13–14: Practical Mastery
## The Capstone — Judgment, Evaluation, and End-to-End ML

---

You now know the mechanics. This final week is about **judgment**:

- When do models fail in the real world (and why)?
- How do you evaluate honestly?
- How do you pick the right model for a problem?
- How do you run a complete ML project from raw data to final answer?

### What we cover:
1. The Evaluation Problem — 3 ways people fool themselves
2. Cross-Validation — getting reliable estimates
3. Detecting and Fixing Overfitting — 3 scenarios
4. Regularization Deep Dive — L1, L2, ElasticNet
5. Model Selection Framework — a decision guide
6. Capstone Project — Breast Cancer dataset, full pipeline
7. Mastery Checklist — self-assessment across all 14 weeks

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import (
    LogisticRegression, Ridge, Lasso, ElasticNet, LinearRegression
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold, LeaveOneOut
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.datasets import make_classification, make_moons, load_breast_cancer
import time

np.random.seed(42)
print('All libraries loaded successfully.')

---
## Part 1: The Evaluation Problem

> **The cardinal sin of ML: evaluating on data you trained on.**

There are three common ways people fool themselves. We will demonstrate each one.

### Bad Example 1: The Overconfident Student

You fit a very complex model (degree-10 polynomial) on 15 data points, then report accuracy on the **same** data. The number looks amazing. But the model has simply memorized the data — it has learned nothing generalizable.

In [ ]:
# Generate 15 data points from a noisy linear relationship
np.random.seed(7)
n = 15
X_small = np.linspace(0, 1, n).reshape(-1, 1)
y_small = 2 * X_small.ravel() + 0.3 * np.random.randn(n)

# Fit a degree-10 polynomial (massively over-parameterized for 15 points)
poly = PolynomialFeatures(degree=10)
X_poly = poly.fit_transform(X_small)

model_overfit = LinearRegression().fit(X_poly, y_small)

# Evaluate on the SAME data (wrong!)
train_preds = model_overfit.predict(X_poly)
train_r2 = model_overfit.score(X_poly, y_small)

# Generate 15 NEW test points
np.random.seed(99)
X_new = np.sort(np.random.uniform(0, 1, 15)).reshape(-1, 1)
y_new = 2 * X_new.ravel() + 0.3 * np.random.randn(15)
X_new_poly = poly.transform(X_new)
test_r2 = model_overfit.score(X_new_poly, y_new)

print(f'Degree-10 polynomial on 15 points:')
print(f'  Train R²  (same data):  {train_r2:.4f}  ← looks perfect!')
print(f'  Test  R²  (new data):   {test_r2:.4f}  ← actual performance')

# Visualize
x_plot = np.linspace(-0.05, 1.05, 300).reshape(-1, 1)
x_plot_poly = poly.transform(x_plot)
y_plot = model_overfit.predict(x_plot_poly)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, title, Xt, yt, color in zip(
    axes,
    ['"Training" data — model was fit on these', 'NEW test data — model has never seen these'],
    [X_small, X_new],
    [y_small, y_new],
    ['steelblue', 'tomato']
):
    ax.plot(x_plot, y_plot, 'k-', lw=1.5, label='Degree-10 model', zorder=1)
    ax.scatter(Xt, yt, color=color, zorder=3, s=60, label='Data points')
    ax.plot(x_plot, 2*x_plot, 'g--', lw=1.5, alpha=0.6, label='True relationship')
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.8, 2.8)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)

axes[0].set_ylabel('y')
fig.suptitle(
    f'Overconfident Model  |  Train R²={train_r2:.2f}  vs  Test R²={test_r2:.2f}',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

print('\nLesson: Training accuracy is a lie. Always evaluate on held-out data.')

### Bad Example 2: The Leaky Pipeline

Data leakage happens when information from the test set accidentally influences training. The most common form: **normalizing the full dataset before splitting**.

When you compute the mean and std of the entire dataset (including test rows) and use those statistics to scale both train and test, the test set statistics have "leaked" into the training process. Your reported accuracy is optimistic.

In [ ]:
# Dataset with deliberately different train/test distributions
np.random.seed(42)
X_leak, y_leak = make_classification(
    n_samples=500, n_features=20, n_informative=5, random_state=42
)

# ── WRONG approach: scale BEFORE splitting ──────────────────────────────────
scaler_bad = StandardScaler()
X_leak_scaled_bad = scaler_bad.fit_transform(X_leak)   # uses ALL 500 rows

X_tr_bad, X_te_bad, y_tr_bad, y_te_bad = train_test_split(
    X_leak_scaled_bad, y_leak, test_size=0.3, random_state=42
)
model_bad = LogisticRegression(max_iter=1000)
model_bad.fit(X_tr_bad, y_tr_bad)
acc_bad = accuracy_score(y_te_bad, model_bad.predict(X_te_bad))

# ── CORRECT approach: scale AFTER splitting ─────────────────────────────────
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_leak, y_leak, test_size=0.3, random_state=42
)
scaler_good = StandardScaler()
X_tr_c = scaler_good.fit_transform(X_tr_c)   # fit on TRAIN only
X_te_c = scaler_good.transform(X_te_c)        # apply same transform to TEST

model_good = LogisticRegression(max_iter=1000)
model_good.fit(X_tr_c, y_tr_c)
acc_good = accuracy_score(y_te_c, model_good.predict(X_te_c))

print('Data Leakage Demonstration')
print('=' * 45)
print(f'  WRONG (scale entire dataset first): {acc_bad:.4f}')
print(f'  CORRECT (scale train only):         {acc_good:.4f}')
print(f'  Inflation from leakage:             {acc_bad - acc_good:+.4f}')

# Visualize the difference in scaler statistics
feature_idx = 0
print(f'\nFor feature 0:')
print(f'  Scaler mean (bad, full data):  {scaler_bad.mean_[feature_idx]:.4f}')
print(f'  Scaler mean (good, train only):{scaler_good.mean_[feature_idx]:.4f}')
print(f'  (They differ because test data shifted the mean)')

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    ['WRONG\n(leak before split)', 'CORRECT\n(split then scale)'],
    [acc_bad, acc_good],
    color=['tomato', 'steelblue'], width=0.4, edgecolor='black'
)
ax.set_ylim(0.8, 0.95)
ax.set_ylabel('Test Accuracy')
ax.set_title('Data Leakage: Inflated vs Honest Accuracy', fontweight='bold')
for bar, val in zip(bars, [acc_bad, acc_good]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nLesson: Always fit preprocessing on TRAIN data only. Apply to test.')

### Bad Example 3: Test Set Exhaustion

If you try 20 models and pick the one with the best **test set** score, you have effectively used the test set as a training signal. The test set is now "used up" — its score is no longer an unbiased estimate of real-world performance.

**The correct three-way split:**
```
Train (60%)      → fit model weights
Validation (20%) → tune hyperparameters, select model
Test (20%)       → FINAL evaluation — touch it ONCE at the very end
```

In [ ]:
# Simulate: try many models, see how test-set selection inflates apparent accuracy
np.random.seed(0)
X_ex, y_ex = make_classification(n_samples=300, n_features=10, random_state=0)

X_train_ex, X_rest, y_train_ex, y_rest = train_test_split(
    X_ex, y_ex, test_size=0.4, random_state=0
)
X_val_ex, X_test_ex, y_val_ex, y_test_ex = train_test_split(
    X_rest, y_rest, test_size=0.5, random_state=0
)

# Try 20 random configurations of LogisticRegression (different C values)
Cs = np.logspace(-3, 3, 20)
val_scores, test_scores = [], []

scaler_ex = StandardScaler()
Xtr = scaler_ex.fit_transform(X_train_ex)
Xvl = scaler_ex.transform(X_val_ex)
Xte = scaler_ex.transform(X_test_ex)

for C in Cs:
    m = LogisticRegression(C=C, max_iter=1000)
    m.fit(Xtr, y_train_ex)
    val_scores.append(accuracy_score(y_val_ex, m.predict(Xvl)))
    test_scores.append(accuracy_score(y_test_ex, m.predict(Xte)))

best_val_idx = np.argmax(val_scores)
best_test_idx = np.argmax(test_scores)   # cheating — looking at test!

print('Test Set Exhaustion Demonstration')
print('=' * 50)
print(f'Best model chosen by VALIDATION: test acc = {test_scores[best_val_idx]:.4f}  (honest)')
print(f'Best model chosen by TEST SET:   test acc = {test_scores[best_test_idx]:.4f}  (cheating)')
print(f'Inflation from peeking at test:  {test_scores[best_test_idx] - test_scores[best_val_idx]:+.4f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.log10(Cs), val_scores, 'o-', color='steelblue', label='Validation accuracy', lw=1.5)
ax.plot(np.log10(Cs), test_scores, 's--', color='tomato', label='Test accuracy (do NOT use for selection)', lw=1.5)
ax.axvline(np.log10(Cs[best_val_idx]), color='steelblue', ls=':', lw=1.5,
           label=f'Val-selected C=10^{np.log10(Cs[best_val_idx]):.1f}')
ax.axvline(np.log10(Cs[best_test_idx]), color='tomato', ls=':', lw=1.5,
           label=f'Test-selected C=10^{np.log10(Cs[best_test_idx]):.1f} (CHEAT)')
ax.set_xlabel('log10(C)')
ax.set_ylabel('Accuracy')
ax.set_title('Selecting on Validation vs Test Set', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print('\nLesson: Use validation for model selection. Test set touched ONCE at the end.')

---
## Part 2: Cross-Validation

A single train/test split can be "lucky" or "unlucky" depending on which samples end up where. Cross-validation gives a more reliable estimate by averaging over multiple splits.

| Method | Best for | Variance |
|---|---|---|
| Single split | Large datasets, fast iteration | High |
| K-Fold CV | Most situations | Low |
| Leave-One-Out | Very small datasets (<50 samples) | Lowest |

In [ ]:
# Dataset for CV comparison
np.random.seed(42)
X_cv, y_cv = make_classification(
    n_samples=150, n_features=10, n_informative=5, random_state=42
)

model_cv = LogisticRegression(max_iter=500)

# ── 1. Single train/test split (repeated 20 times to show variance) ──────────
single_scores = []
for seed in range(20):
    Xtr, Xte, ytr, yte = train_test_split(X_cv, y_cv, test_size=0.3, random_state=seed)
    sc = StandardScaler()
    model_cv.fit(sc.fit_transform(Xtr), ytr)
    single_scores.append(accuracy_score(yte, model_cv.predict(sc.transform(Xte))))

# ── 2. 5-fold CV ─────────────────────────────────────────────────────────────
pipe_cv = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=500))])
fold5_scores = cross_val_score(pipe_cv, X_cv, y_cv, cv=5, scoring='accuracy')

# ── 3. Leave-One-Out CV ───────────────────────────────────────────────────────
loo_scores = cross_val_score(pipe_cv, X_cv, y_cv, cv=LeaveOneOut(), scoring='accuracy')

print('Cross-Validation Comparison')
print('=' * 50)
print(f'Single split (20 runs):  mean={np.mean(single_scores):.3f}  std={np.std(single_scores):.3f}')
print(f'5-Fold CV:               mean={np.mean(fold5_scores):.3f}  std={np.std(fold5_scores):.3f}')
print(f'Leave-One-Out:           mean={np.mean(loo_scores):.3f}  std={np.std(loo_scores):.3f}')
print(f'\n5-Fold scores by fold: {fold5_scores.round(3)}')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].hist(single_scores, bins=8, color='steelblue', edgecolor='white')
axes[0].axvline(np.mean(single_scores), color='red', ls='--', lw=2, label=f'Mean={np.mean(single_scores):.3f}')
axes[0].set_title(f'20× Single Splits\nstd={np.std(single_scores):.3f}', fontweight='bold')
axes[0].set_xlabel('Accuracy'); axes[0].legend()

axes[1].bar(range(1, 6), fold5_scores, color='seagreen', edgecolor='black')
axes[1].axhline(np.mean(fold5_scores), color='red', ls='--', lw=2, label=f'Mean={np.mean(fold5_scores):.3f}')
axes[1].set_title(f'5-Fold CV Scores\nstd={np.std(fold5_scores):.3f}', fontweight='bold')
axes[1].set_xlabel('Fold'); axes[1].set_ylabel('Accuracy'); axes[1].legend()
axes[1].set_ylim(0.7, 1.0)

axes[2].hist(loo_scores, bins=10, color='darkorange', edgecolor='white')
axes[2].axvline(np.mean(loo_scores), color='red', ls='--', lw=2, label=f'Mean={np.mean(loo_scores):.3f}')
axes[2].set_title(f'Leave-One-Out (n=150)\nstd={np.std(loo_scores):.3f}', fontweight='bold')
axes[2].set_xlabel('Accuracy'); axes[2].legend()

plt.suptitle('Three CV Methods — Accuracy Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Model comparison via CV — box plot of fold scores
np.random.seed(42)
X_cmp, y_cmp = make_classification(
    n_samples=300, n_features=15, n_informative=6, random_state=42
)

models_cmp = {
    'Logistic\nRegression': Pipeline([('sc', StandardScaler()),
                                       ('lr', LogisticRegression(max_iter=500))]),
    'Decision Tree\n(depth=3)': DecisionTreeClassifier(max_depth=3, random_state=0),
    'Decision Tree\n(no limit)': DecisionTreeClassifier(random_state=0),
    'Random Forest\n(n=10)': RandomForestClassifier(n_estimators=10, random_state=0),
}

cv_results = {}
for name, mdl in models_cmp.items():
    scores = cross_val_score(mdl, X_cmp, y_cmp, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f'{name.replace(chr(10)," "):30s}  mean={scores.mean():.3f}  std={scores.std():.3f}')

fig, ax = plt.subplots(figsize=(9, 5))
data_box = list(cv_results.values())
labels_box = list(cv_results.keys())
bp = ax.boxplot(data_box, labels=labels_box, patch_artist=True)
colors_box = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('5-Fold CV Accuracy')
ax.set_title('Model Comparison via 5-Fold CV\n(box = IQR of fold scores)', fontweight='bold')
ax.yaxis.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print('\nKey insight: Decision Tree (no limit) has high variance — sign of overfitting.')

---
## Part 3: Detecting and Fixing Overfitting

Overfitting = the model learns the training data too well, including its noise, and fails to generalize.

**Signatures of overfitting:**
- Training accuracy >> Test/Validation accuracy
- Validation loss increases while training loss decreases
- Model is very complex relative to the amount of data

### Scenario A: Polynomial Overfitting

As polynomial degree increases, the model becomes more flexible. At some point it starts fitting the noise.

In [ ]:
# Generate regression data from a cubic + noise
np.random.seed(5)
n_poly = 40
X_p = np.sort(np.random.uniform(0, 1, n_poly))
y_p = np.sin(2 * np.pi * X_p) + 0.3 * np.random.randn(n_poly)

X_p_tr, X_p_te, y_p_tr, y_p_te = train_test_split(
    X_p.reshape(-1, 1), y_p, test_size=0.3, random_state=0
)

degrees = [1, 3, 7, 12]
train_errs, test_errs = [], []

fig_top, axes_top = plt.subplots(1, 4, figsize=(15, 4), sharey=True)
x_line = np.linspace(0, 1, 300).reshape(-1, 1)

for ax, deg in zip(axes_top, degrees):
    poly_f = PolynomialFeatures(degree=deg)
    lr = LinearRegression()
    lr.fit(poly_f.fit_transform(X_p_tr), y_p_tr)
    
    tr_err = np.mean((y_p_tr - lr.predict(poly_f.transform(X_p_tr)))**2)
    te_err = np.mean((y_p_te - lr.predict(poly_f.transform(X_p_te)))**2)
    train_errs.append(tr_err)
    test_errs.append(te_err)
    
    y_line = lr.predict(poly_f.transform(x_line))
    ax.scatter(X_p_tr, y_p_tr, color='steelblue', s=30, label='Train', zorder=3)
    ax.scatter(X_p_te, y_p_te, color='tomato', s=30, label='Test', zorder=3)
    ax.plot(x_line, y_line, 'k-', lw=2)
    ax.plot(x_line, np.sin(2*np.pi*x_line), 'g--', lw=1, alpha=0.6, label='True')
    ax.set_title(f'Degree {deg}\nTrain MSE={tr_err:.3f}  Test MSE={te_err:.3f}', fontsize=9)
    ax.set_ylim(-2.5, 2.5)
    if deg == 1:
        ax.legend(fontsize=8)

plt.suptitle('Polynomial Degree vs Fit Quality', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Train vs test error vs degree
fig2, ax2 = plt.subplots(figsize=(7, 4))
ax2.plot(degrees, train_errs, 'o-', color='steelblue', label='Train MSE', lw=2)
ax2.plot(degrees, test_errs, 's--', color='tomato', label='Test MSE', lw=2)
ax2.set_xlabel('Polynomial Degree')
ax2.set_ylabel('Mean Squared Error')
ax2.set_title('Bias-Variance Tradeoff: Train vs Test Error', fontweight='bold')
ax2.legend()
ax2.set_yscale('log')
plt.tight_layout()
plt.show()

print('As degree increases: train error falls, but test error eventually rises.')
print('Fix: choose degree by test/CV error, not training error.')

### Scenario B: Decision Tree Overfitting

An unlimited decision tree can achieve 100% training accuracy by memorizing every sample. Fixing it: limit `max_depth` (pruning).

In [ ]:
np.random.seed(42)
X_dt, y_dt = make_moons(n_samples=200, noise=0.3, random_state=42)
X_dt_tr, X_dt_te, y_dt_tr, y_dt_te = train_test_split(
    X_dt, y_dt, test_size=0.3, random_state=0
)

# Unlimited depth
dt_full = DecisionTreeClassifier(random_state=0)
dt_full.fit(X_dt_tr, y_dt_tr)
print('Unlimited depth tree:')
print(f'  Training accuracy:   {dt_full.score(X_dt_tr, y_dt_tr):.3f}')
print(f'  Test accuracy:       {dt_full.score(X_dt_te, y_dt_te):.3f}')
print(f'  Tree depth:          {dt_full.get_depth()}')

# Tuning max_depth via cross-validation
depths = range(1, 11)
cv_mean, cv_std, tr_acc = [], [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=0)
    s = cross_val_score(dt, X_dt_tr, y_dt_tr, cv=5)
    cv_mean.append(s.mean())
    cv_std.append(s.std())
    dt.fit(X_dt_tr, y_dt_tr)
    tr_acc.append(dt.score(X_dt_tr, y_dt_tr))

best_depth = depths[np.argmax(cv_mean)]
print(f'\nBest depth by CV: {best_depth}')

dt_best = DecisionTreeClassifier(max_depth=best_depth, random_state=0)
dt_best.fit(X_dt_tr, y_dt_tr)
print(f'  Test accuracy at depth={best_depth}: {dt_best.score(X_dt_te, y_dt_te):.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
cv_mean_arr = np.array(cv_mean)
cv_std_arr = np.array(cv_std)
ax.plot(list(depths), tr_acc, 'o-', color='steelblue', lw=2, label='Train accuracy')
ax.plot(list(depths), cv_mean_arr, 's-', color='tomato', lw=2, label='CV accuracy (mean)')
ax.fill_between(list(depths),
                cv_mean_arr - cv_std_arr,
                cv_mean_arr + cv_std_arr,
                alpha=0.2, color='tomato', label='CV ±1 std')
ax.axvline(best_depth, ls='--', color='green', lw=2, label=f'Best depth={best_depth}')
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree: Finding Optimal Depth via CV', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### Scenario C: Neural Network Overfitting

A large network on a small dataset will overfit. We track training and validation loss across iterations to visualize the overfitting signature.

In [ ]:
# Small dataset — deliberately underpowered to induce overfitting
np.random.seed(42)
X_nn, y_nn = make_classification(
    n_samples=150, n_features=20, n_informative=5, random_state=42
)
X_nn_tr, X_nn_val, y_nn_tr, y_nn_val = train_test_split(
    X_nn, y_nn, test_size=0.3, random_state=0
)

sc_nn = StandardScaler()
X_nn_tr_s = sc_nn.fit_transform(X_nn_tr)
X_nn_val_s = sc_nn.transform(X_nn_val)

# Use warm_start to track loss epoch-by-epoch
mlp = MLPClassifier(
    hidden_layer_sizes=(100, 100, 100),  # 3 hidden layers — large for this dataset
    max_iter=1, warm_start=True, random_state=42, learning_rate_init=0.01
)

train_losses, val_losses = [], []
train_accs, val_accs = [], []

n_epochs = 150
for epoch in range(n_epochs):
    mlp.fit(X_nn_tr_s, y_nn_tr)
    train_losses.append(mlp.loss_)
    val_losses.append(
        -np.mean(np.log(np.clip(
            mlp.predict_proba(X_nn_val_s)[np.arange(len(y_nn_val)), y_nn_val], 1e-10, 1
        )))
    )
    train_accs.append(mlp.score(X_nn_tr_s, y_nn_tr))
    val_accs.append(mlp.score(X_nn_val_s, y_nn_val))

best_epoch = np.argmin(val_losses)
print(f'Best validation epoch: {best_epoch+1}')
print(f'  Train accuracy at best epoch: {train_accs[best_epoch]:.3f}')
print(f'  Val   accuracy at best epoch: {val_accs[best_epoch]:.3f}')
print(f'  Final train accuracy:         {train_accs[-1]:.3f}')
print(f'  Final val   accuracy:         {val_accs[-1]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train loss', color='steelblue')
axes[0].plot(val_losses, label='Validation loss', color='tomato')
axes[0].axvline(best_epoch, ls='--', color='green', lw=1.5, label=f'Early stop (epoch {best_epoch+1})')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('Loss Curves — Overfitting Signature', fontweight='bold')
axes[0].legend()

axes[1].plot(train_accs, label='Train accuracy', color='steelblue')
axes[1].plot(val_accs, label='Validation accuracy', color='tomato')
axes[1].axvline(best_epoch, ls='--', color='green', lw=1.5, label=f'Early stop (epoch {best_epoch+1})')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curves', fontweight='bold')
axes[1].legend()

plt.suptitle('MLP Overfitting: Large Network on Small Dataset (n=150)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nFixes: early stopping | smaller network | more data | dropout (PyTorch/TF)')

---
## Part 4: Regularization Deep Dive

Regularization adds a penalty to the loss function to discourage large weights, reducing overfitting.

| Method | Penalty | Effect |
|---|---|---|
| Ridge (L2) | sum of squared weights | Shrinks weights smoothly toward 0 |
| Lasso (L1) | sum of absolute weights | Zeroes out irrelevant features |
| ElasticNet | L1 + L2 combined | Both effects: sparsity + smoothness |

In [ ]:
# Dataset: 10 features, only 3 are truly informative
np.random.seed(42)
n_reg = 80
n_features_reg = 10
n_informative_reg = 3

# True weights: first 3 non-zero, rest are 0
true_coef = np.array([2.5, -1.8, 1.2] + [0.0] * 7)
X_reg = np.random.randn(n_reg, n_features_reg)
y_reg = X_reg @ true_coef + 0.5 * np.random.randn(n_reg)

X_reg_tr, X_reg_te, y_reg_tr, y_reg_te = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=0
)

alpha = 0.1
models_reg = {
    'No Reg\n(Linear)': LinearRegression(),
    f'Ridge (L2)\nalpha={alpha}': Ridge(alpha=alpha),
    f'Lasso (L1)\nalpha={alpha}': Lasso(alpha=alpha, max_iter=5000),
    f'ElasticNet\nalpha={alpha}': ElasticNet(alpha=alpha, l1_ratio=0.5, max_iter=5000),
}

print(f'True informative features: 0, 1, 2  (true coefs: {true_coef[:3]})')
print(f'Noise features: 3–9  (true coefs: all 0)')
print()
print(f'{"Model":<22} {"Train MSE":>10} {"Test MSE":>10}  Coefficients')
print('-' * 75)

coef_dict = {}
for name, mdl in models_reg.items():
    mdl.fit(X_reg_tr, y_reg_tr)
    tr_mse = np.mean((y_reg_tr - mdl.predict(X_reg_tr))**2)
    te_mse = np.mean((y_reg_te - mdl.predict(X_reg_te))**2)
    coef_dict[name] = mdl.coef_
    label = name.replace('\n', ' ')
    print(f'{label:<22} {tr_mse:>10.3f} {te_mse:>10.3f}  {mdl.coef_.round(2)}')

# Coefficient comparison plot
fig, axes = plt.subplots(1, 4, figsize=(15, 4), sharey=True)
feature_labels = [f'F{i}' for i in range(10)]
colors_coef = ['#aec7e8', '#ffbb78', '#98df8a', '#c5b0d5']

for ax, (name, coefs), col in zip(axes, coef_dict.items(), colors_coef):
    bars = ax.bar(feature_labels, coefs, color=col, edgecolor='black', linewidth=0.5)
    # Highlight informative features
    for i in range(3):
        bars[i].set_edgecolor('red')
        bars[i].set_linewidth(2)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(name, fontsize=9, fontweight='bold')
    ax.set_xticklabels(feature_labels, rotation=45, fontsize=8)

axes[0].set_ylabel('Coefficient value')
plt.suptitle(
    'Regularization Comparison — Red borders = true informative features',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

print('\nLasso zeros out noise features (F3–F9). Ridge shrinks them but keeps them.')

In [ ]:
# Regularization path: how coefficients change as alpha increases
alphas = np.logspace(-2, 2, 60)

ridge_coefs = []
lasso_coefs = []

for a in alphas:
    r = Ridge(alpha=a).fit(X_reg_tr, y_reg_tr)
    ridge_coefs.append(r.coef_)
    l = Lasso(alpha=a, max_iter=5000).fit(X_reg_tr, y_reg_tr)
    lasso_coefs.append(l.coef_)

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
feature_colors = plt.cm.tab10(np.linspace(0, 1, 10))

for ax, coefs, title, method in zip(
    axes, [ridge_coefs, lasso_coefs],
    ['Ridge (L2) Regularization Path', 'Lasso (L1) Regularization Path'],
    ['Ridge', 'Lasso']
):
    for i in range(10):
        ls = '-' if i < 3 else '--'
        lw = 2.5 if i < 3 else 1.0
        label = f'F{i} (informative)' if i < 3 else f'F{i} (noise)'
        ax.plot(np.log10(alphas), coefs[:, i], ls=ls, lw=lw,
                color=feature_colors[i], label=label if i < 3 or i == 3 else '')
    ax.axhline(0, color='black', lw=0.8, ls=':')
    ax.set_xlabel('log10(alpha) — regularization strength →')
    ax.set_ylabel('Coefficient value')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Regularization Paths: Solid=informative features, Dashed=noise',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Lasso: noise features hit exactly 0 at moderate alpha.')
print('Ridge: all coefficients shrink but rarely reach exactly 0.')

---
## Part 5: Model Selection Framework

There is no single best model. The right choice depends on your data size, feature count, need for interpretability, and whether the relationship is linear.

```
How many samples?
  < 100     → Simple model (linear, small tree). CV is essential.
  100–10k   → Try several models, use CV for comparison.
  > 10k     → Gradient boosting or deep learning are options.

How many features?
  < 10      → Linear model + polynomial may suffice.
  10–100    → Random forest / gradient boosting.
  > 100     → Feature selection or dimensionality reduction first.

Is the relationship linear?
  Yes → Logistic/Linear regression (fast, interpretable).
  No  → Tree-based or neural network.

Do you need interpretability?
  Yes → Linear model (inspect weights) or shallow tree (read rules).
  No  → Any model, optimize for accuracy.
```

In [ ]:
# Compare 6 models × 3 datasets — accuracy heatmap
from sklearn.datasets import make_classification

# Three datasets with different characteristics
datasets = {
    'Small, Linear\n(n=100, 5 feat)': make_classification(
        n_samples=100, n_features=5, n_informative=4, n_redundant=0, random_state=1
    ),
    'Medium, Non-linear\n(n=1000, moons)': make_moons(n_samples=1000, noise=0.2, random_state=42),
    'Large, Complex\n(n=3000, 20 feat)': make_classification(
        n_samples=3000, n_features=20, n_informative=10, n_redundant=4, random_state=7
    ),
}

# Six models
models_sel = {
    'Logistic Reg': Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=500))]),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=0),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=0),
    'Grad. Boosting': GradientBoostingClassifier(n_estimators=50, random_state=0),
    'MLP': Pipeline([('sc', StandardScaler()), ('mlp', MLPClassifier(hidden_layer_sizes=(50,), max_iter=300, random_state=0))]),
    'KNN': Pipeline([('sc', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))]),
}

acc_matrix = np.zeros((len(datasets), len(models_sel)))
time_matrix = np.zeros_like(acc_matrix)

print(f'{"":<30}', '  '.join(f'{k[:10]:>12}' for k in models_sel))
for i, (ds_name, (X_ds, y_ds)) in enumerate(datasets.items()):
    row_accs = []
    for j, (m_name, mdl) in enumerate(models_sel.items()):
        t0 = time.time()
        scores = cross_val_score(mdl, X_ds, y_ds, cv=3, scoring='accuracy')
        elapsed = time.time() - t0
        acc_matrix[i, j] = scores.mean()
        time_matrix[i, j] = elapsed
        row_accs.append(f'{scores.mean():.3f}')
    print(f'{ds_name.replace(chr(10)," "):<30}', '  '.join(f'{v:>12}' for v in row_accs))

# Heatmap
fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(acc_matrix, cmap='RdYlGn', vmin=0.6, vmax=1.0, aspect='auto')
ax.set_xticks(range(len(models_sel)))
ax.set_xticklabels(list(models_sel.keys()), rotation=20, ha='right', fontsize=10)
ax.set_yticks(range(len(datasets)))
ax.set_yticklabels([k.replace('\n', ' ') for k in datasets.keys()], fontsize=9)

for i in range(acc_matrix.shape[0]):
    for j in range(acc_matrix.shape[1]):
        ax.text(j, i, f'{acc_matrix[i,j]:.2f}', ha='center', va='center',
                fontsize=11, fontweight='bold', color='black')

plt.colorbar(im, ax=ax, label='CV Accuracy')
ax.set_title('Model × Dataset Accuracy Heatmap (3-fold CV)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nObservation: no single model wins on all datasets.')
print('Logistic Reg is competitive on linear data; ensemble methods shine on complex data.')

---
## Part 6: Capstone Project — Breast Cancer Wisconsin Dataset

We now run a **complete, production-quality ML pipeline** from raw data to final evaluation. Every design choice is explained.

This is the workflow you will follow in real projects.

### Step 1: Exploratory Data Analysis (EDA)

Before touching a model, understand your data: size, class balance, feature distributions, correlations.

In [ ]:
# Load dataset
cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target   # 0 = malignant, 1 = benign
feature_names = cancer.feature_names
target_names = cancer.target_names

print('=== Breast Cancer Wisconsin Dataset ===')
print(f'Shape:          {X_cancer.shape}  ({X_cancer.shape[0]} samples, {X_cancer.shape[1]} features)')
print(f'Classes:        {list(target_names)}  (0=malignant, 1=benign)')
n_malig = np.sum(y_cancer == 0)
n_benign = np.sum(y_cancer == 1)
print(f'Class balance:  {n_malig} malignant ({n_malig/len(y_cancer)*100:.1f}%)'
      f'  /  {n_benign} benign ({n_benign/len(y_cancer)*100:.1f}%)')

# Feature statistics (top 5 by mean absolute value)
feat_means = np.abs(np.mean(X_cancer, axis=0))
top5_idx = np.argsort(feat_means)[::-1][:5]
print('\nTop 5 features by mean magnitude:')
print(f'{"Feature":<35} {"Mean":>10} {"Std":>10}')
for idx in top5_idx:
    print(f'{feature_names[idx]:<35} {np.mean(X_cancer[:, idx]):>10.2f} {np.std(X_cancer[:, idx]):>10.2f}')

In [ ]:
# Correlation heatmap — top 10 features
top10_idx = np.argsort(feat_means)[::-1][:10]
X_top10 = X_cancer[:, top10_idx]
top10_names = [feature_names[i][:18] for i in top10_idx]

corr_matrix = np.corrcoef(X_top10.T)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(top10_names, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(top10_names, fontsize=8)
for i in range(10):
    for j in range(10):
        ax.text(j, i, f'{corr_matrix[i,j]:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_title('Correlation Heatmap — Top 10 Features', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution plots: benign vs malignant for 4 features
plot_features = [0, 2, 3, 7]  # mean radius, mean perimeter, mean area, mean concave points

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, fi in zip(axes, plot_features):
    vals_malig = X_cancer[y_cancer == 0, fi]
    vals_benign = X_cancer[y_cancer == 1, fi]
    ax.hist(vals_malig, bins=20, alpha=0.6, color='tomato', density=True, label='Malignant')
    ax.hist(vals_benign, bins=20, alpha=0.6, color='steelblue', density=True, label='Benign')
    ax.set_title(feature_names[fi][:22], fontsize=9, fontweight='bold')
    ax.set_xlabel('Value'); ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions: Benign vs Malignant', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Observation: malignant tumors tend to have larger radius, perimeter, and area.')

### Step 2: Preprocessing

Split first, then scale. The scaler sees only training data.

In [ ]:
# Three-way split: 60% train / 20% validation / 20% test
X_tr_cap, X_rest_cap, y_tr_cap, y_rest_cap = train_test_split(
    X_cancer, y_cancer, test_size=0.40, random_state=42, stratify=y_cancer
)
X_val_cap, X_te_cap, y_val_cap, y_te_cap = train_test_split(
    X_rest_cap, y_rest_cap, test_size=0.50, random_state=42, stratify=y_rest_cap
)

print('Split sizes:')
print(f'  Train:      {X_tr_cap.shape[0]} samples ({X_tr_cap.shape[0]/len(y_cancer)*100:.0f}%)')
print(f'  Validation: {X_val_cap.shape[0]} samples ({X_val_cap.shape[0]/len(y_cancer)*100:.0f}%)')
print(f'  Test:       {X_te_cap.shape[0]} samples ({X_te_cap.shape[0]/len(y_cancer)*100:.0f}%)')

# Fit scaler on TRAIN ONLY
scaler_cap = StandardScaler()
X_tr_sc = scaler_cap.fit_transform(X_tr_cap)    # learn mean/std from train
X_val_sc = scaler_cap.transform(X_val_cap)       # apply same transform
X_te_sc  = scaler_cap.transform(X_te_cap)        # apply same transform

print('\nScaler fit on train only. Same transformation applied to val + test.')
print(f'Scaler mean (feature 0): {scaler_cap.mean_[0]:.4f}  (from train data only)')

### Step 3: Baseline Model — Logistic Regression

Start with the simplest model. If logistic regression achieves 95% accuracy, you may not need anything more complex.

In [ ]:
# Baseline: logistic regression
lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_base.fit(X_tr_sc, y_tr_cap)
y_val_pred_base = lr_base.predict(X_val_sc)

print('=== Baseline: Logistic Regression (Validation Set) ===')
print(f'Accuracy:  {accuracy_score(y_val_cap, y_val_pred_base):.4f}')
print(f'Precision: {precision_score(y_val_cap, y_val_pred_base):.4f}')
print(f'Recall:    {recall_score(y_val_cap, y_val_pred_base):.4f}')
print(f'F1:        {f1_score(y_val_cap, y_val_pred_base):.4f}')

cm = confusion_matrix(y_val_cap, y_val_pred_base)
print('\nConfusion Matrix:')
print(f'  Predicted →      Malignant   Benign')
print(f'  Actual Malignant   {cm[0,0]:4d}      {cm[0,1]:4d}')
print(f'  Actual Benign      {cm[1,0]:4d}      {cm[1,1]:4d}')

# Top 5 weighted features
coef_lr = lr_base.coef_[0]
top5_abs = np.argsort(np.abs(coef_lr))[::-1][:5]
print('\nTop 5 features by weight magnitude:')
for idx in top5_abs:
    direction = 'benign' if coef_lr[idx] > 0 else 'malignant'
    print(f'  {feature_names[idx]:<35} coef={coef_lr[idx]:+.3f}  → predicts {direction}')

# Confusion matrix plot
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred Malignant', 'Pred Benign'])
ax.set_yticklabels(['True Malignant', 'True Benign'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=16, fontweight='bold')
plt.colorbar(im, ax=ax)
ax.set_title('Confusion Matrix — Logistic Regression (Val)', fontweight='bold')
plt.tight_layout()
plt.show()

### Step 4: Model Comparison on Validation Set

Try multiple models, compare on validation set, confirm with CV.

In [ ]:
# Compare 4 models on validation set and via CV
models_cap = {
    'Logistic Reg': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree (d=3)': DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42),
}

val_accs_cap = {}
cv_accs_cap = {}

print(f'{"Model":<22} {"Val Accuracy":>14} {"CV Mean":>10} {"CV Std":>8}')
print('-' * 58)

for name, mdl in models_cap.items():
    mdl.fit(X_tr_sc, y_tr_cap)
    v_acc = accuracy_score(y_val_cap, mdl.predict(X_val_sc))
    val_accs_cap[name] = v_acc
    # CV on combined train+val for more stable estimate
    X_tv = np.vstack([X_tr_sc, X_val_sc])
    y_tv = np.concatenate([y_tr_cap, y_val_cap])
    cv_s = cross_val_score(mdl, X_tv, y_tv, cv=5)
    cv_accs_cap[name] = cv_s
    print(f'{name:<22} {v_acc:>14.4f} {cv_s.mean():>10.4f} {cv_s.std():>8.4f}')

best_model_name = max(cv_accs_cap, key=lambda k: cv_accs_cap[k].mean())
print(f'\nBest model by CV: {best_model_name}')

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
names_cap = list(cv_accs_cap.keys())
means_cap = [v.mean() for v in cv_accs_cap.values()]
stds_cap  = [v.std()  for v in cv_accs_cap.values()]
colors_cap = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
bars_cap = ax.bar(names_cap, means_cap, color=colors_cap, edgecolor='black',
                  yerr=stds_cap, capsize=5)
ax.set_ylim(0.90, 1.01)
ax.set_ylabel('5-Fold CV Accuracy')
ax.set_title('Model Comparison — Breast Cancer Dataset (CV ± std)', fontweight='bold')
for bar, v in zip(bars_cap, means_cap):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{v:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

### Step 5: Hyperparameter Tuning

For Random Forest (best model), tune `n_estimators`. We use validation accuracy to avoid leaking into the test set.

In [ ]:
# Tune n_estimators for Random Forest
n_est_values = [5, 10, 20, 30, 50, 75, 100, 150, 200]
cv_scores_ne = []
val_scores_ne = []

X_tv = np.vstack([X_tr_sc, X_val_sc])
y_tv = np.concatenate([y_tr_cap, y_val_cap])

for ne in n_est_values:
    rf = RandomForestClassifier(n_estimators=ne, random_state=42)
    # Validation score
    rf.fit(X_tr_sc, y_tr_cap)
    val_scores_ne.append(rf.score(X_val_sc, y_val_cap))
    # CV score
    cv = cross_val_score(rf, X_tv, y_tv, cv=5).mean()
    cv_scores_ne.append(cv)

best_ne_idx = np.argmax(cv_scores_ne)
best_ne = n_est_values[best_ne_idx]
print(f'Best n_estimators by CV: {best_ne}  (CV accuracy: {cv_scores_ne[best_ne_idx]:.4f})')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_est_values, val_scores_ne, 'o-', color='steelblue', lw=2, label='Validation accuracy')
ax.plot(n_est_values, cv_scores_ne, 's--', color='tomato', lw=2, label='CV accuracy (mean)')
ax.axvline(best_ne, ls=':', color='green', lw=2, label=f'Best n={best_ne}')
ax.set_xlabel('n_estimators')
ax.set_ylabel('Accuracy')
ax.set_title('Random Forest: Tuning n_estimators', fontweight='bold')
ax.legend()
ax.set_ylim(0.93, 1.005)
plt.tight_layout()
plt.show()

### Step 6: Final Evaluation on Test Set

We now touch the test set **for the first and only time**. The model is retrained on train+validation combined.

In [ ]:
# Retrain on train + validation combined, evaluate on test set
X_trainval = np.vstack([X_tr_sc, X_val_sc])
y_trainval  = np.concatenate([y_tr_cap, y_val_cap])

final_model = RandomForestClassifier(n_estimators=best_ne, random_state=42)
final_model.fit(X_trainval, y_trainval)

y_test_pred_final = final_model.predict(X_te_sc)

final_acc  = accuracy_score(y_te_cap, y_test_pred_final)
final_prec = precision_score(y_te_cap, y_test_pred_final)
final_rec  = recall_score(y_te_cap, y_test_pred_final)
final_f1   = f1_score(y_te_cap, y_test_pred_final)

print('===================================================================')
print('  FINAL TEST SET EVALUATION  (touched once, at the very end)')
print('===================================================================')
print(f'  Model:     Random Forest (n_estimators={best_ne})')
print(f'  Accuracy:  {final_acc:.4f}')
print(f'  Precision: {final_prec:.4f}')
print(f'  Recall:    {final_rec:.4f}')
print(f'  F1 Score:  {final_f1:.4f}')
print('===================================================================')
print()
print('This is our honest estimate of real-world performance.')
print('It is unbiased because the test set was never used during model development.')

cm_final = confusion_matrix(y_te_cap, y_test_pred_final)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Confusion matrix
im = axes[0].imshow(cm_final, cmap='Blues')
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Pred Malignant', 'Pred Benign'])
axes[0].set_yticklabels(['True Malignant', 'True Benign'])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm_final[i, j]), ha='center', va='center',
                     fontsize=18, fontweight='bold')
plt.colorbar(im, ax=axes[0])
axes[0].set_title('Final Confusion Matrix (Test Set)', fontweight='bold')

# Metrics bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
values = [final_acc, final_prec, final_rec, final_f1]
colors_fin = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
bars_fin = axes[1].bar(metrics, values, color=colors_fin, edgecolor='black')
axes[1].set_ylim(0.9, 1.01)
axes[1].set_ylabel('Score')
axes[1].set_title('Final Test Set Metrics', fontweight='bold')
for bar, v in zip(bars_fin, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{v:.3f}', ha='center', fontweight='bold')

plt.suptitle(f'Random Forest (n={best_ne}) — Breast Cancer Final Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Step 7: Insights — What Did the Model Learn?

In [ ]:
# Feature importances from Random Forest
importances = final_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

print('Top 10 most important features (Random Forest importance):')
print(f'{"Rank":<6} {"Feature":<35} {"Importance":>12}')
print('-' * 55)
for rank, idx in enumerate(sorted_idx[:10], 1):
    print(f'{rank:<6} {feature_names[idx]:<35} {importances[idx]:>12.4f}')

fig, ax = plt.subplots(figsize=(9, 6))
top15 = sorted_idx[:15]
ax.barh(
    range(15),
    importances[top15][::-1],
    color='steelblue', edgecolor='black'
)
ax.set_yticks(range(15))
ax.set_yticklabels([feature_names[i][:30] for i in top15[::-1]], fontsize=9)
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest Feature Importances — Top 15', fontweight='bold')
ax.axvline(np.mean(importances), ls='--', color='red', lw=1.5, label='Mean importance')
ax.legend()
plt.tight_layout()
plt.show()

print('\nInsights:')
print(f'  - Top feature: "{feature_names[sorted_idx[0]]}"')
print('  - Worst-case error: false negatives (malignant predicted as benign).')
print('  - To improve: gather more malignant samples; tune decision threshold for recall.')
print('  - To productionize: add monitoring for data drift in top features.')

---
## Part 7: Final Mastery Checklist

Use this to assess your readiness after 14 weeks of study.

---

### Conceptual Understanding

- [ ] I can explain the bias-variance tradeoff in my own words
- [ ] I understand why training accuracy is not a reliable metric
- [ ] I can explain what a p-value means (and what it does NOT mean)
- [ ] I understand what a probability distribution is and why it matters
- [ ] I can explain what a linear model assumes about the data
- [ ] I understand what a decision tree does at each node
- [ ] I can explain what "ensemble" means and why it usually helps
- [ ] I understand gradient descent conceptually ("roll downhill to minimize loss")
- [ ] I can explain what regularization does and why it helps overfitting
- [ ] I understand what cross-entropy loss measures

---

### Technical Skills

- [ ] I can split data into train/validation/test with `train_test_split`
- [ ] I know to fit preprocessors on train data only
- [ ] I can run k-fold cross-validation with `cross_val_score`
- [ ] I can build and evaluate: LogisticRegression, DecisionTree, RandomForest, MLP
- [ ] I can compute and interpret: accuracy, precision, recall, F1, confusion matrix
- [ ] I can apply Ridge, Lasso, ElasticNet and understand the effect
- [ ] I can plot learning curves (train vs val error across model complexity)
- [ ] I can tune a hyperparameter using a validation set or CV
- [ ] I can use sklearn Pipelines to avoid data leakage
- [ ] I can visualize feature importance from a trained model

---

### Judgment

- [ ] Given a new dataset, I know what EDA to do first
- [ ] I can recognize overfitting from train vs val accuracy gap
- [ ] I know when to use logistic regression vs random forest vs MLP
- [ ] I can explain why a model failed and suggest a fix
- [ ] I check for data leakage before trusting any results
- [ ] I keep the test set truly held out until the final evaluation
- [ ] I report uncertainty (e.g., CV std) alongside point estimates
- [ ] I can identify which features a model relies on most
- [ ] I know that class imbalance affects accuracy and use F1/recall instead
- [ ] I can articulate the business cost of false positives vs false negatives

---

### Mindset

- [ ] I start with the simplest model and add complexity only if needed
- [ ] I document my preprocessing steps so they are reproducible
- [ ] I am skeptical of results that seem too good
- [ ] I understand that ML models are not magic — they find statistical patterns
- [ ] I know what questions to ask before choosing a model
- [ ] I treat ML as an iterative process: experiment, measure, improve

---

## What's Next?

You have completed the foundations. Here are concrete next steps:

| Track | First step |
|---|---|
| **Deep Learning** | PyTorch `nn.Module` tutorial → MNIST classifier |
| **Production ML** | sklearn Pipelines → MLflow experiment tracking |
| **Advanced sklearn** | GridSearchCV, ColumnTransformer, FeatureUnion |
| **NLP** | HuggingFace `transformers` → sentiment classification |
| **Computer Vision** | PyTorch / TensorFlow → image classification |
| **Statistics depth** | Bayesian inference, A/B testing, causal inference |

**The single most important habit:** build something. Pick a dataset you care about and run the full pipeline — EDA, preprocessing, modeling, evaluation, insights. Do it again. Do it faster. Do it better.

> *"All models are wrong, but some are useful."* — George Box

You now know how to build useful ones.

In [ ]:
# Self-scoring visualization — check off your skills
categories = {
    'Conceptual\nUnderstanding': 10,
    'Technical\nSkills': 10,
    'Judgment': 10,
    'Mindset': 6,
}

# Example self-scores — update these with your honest assessment (0 to max)
your_scores = {
    'Conceptual\nUnderstanding': 8,   # <-- update this
    'Technical\nSkills': 7,           # <-- update this
    'Judgment': 6,                    # <-- update this
    'Mindset': 5,                     # <-- update this
}

fig, ax = plt.subplots(figsize=(9, 5))
cats = list(categories.keys())
maxes = list(categories.values())
scores = [your_scores[c] for c in cats]
pcts = [s/m*100 for s, m in zip(scores, maxes)]

x = np.arange(len(cats))
ax.bar(x, maxes, color='#e0e0e0', edgecolor='black', width=0.5, label='Max possible')
bars = ax.bar(x, scores, color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'],
              edgecolor='black', width=0.5, alpha=0.85, label='Your score')
ax.set_xticks(x)
ax.set_xticklabels(cats, fontsize=11)
ax.set_ylabel('Items checked')
ax.set_title('14-Week Mastery Self-Assessment\n(Update your_scores above with honest values)',
             fontweight='bold')
for bar, s, m, p in zip(bars, scores, maxes, pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{s}/{m}\n{p:.0f}%', ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0, 13)
ax.legend()
plt.tight_layout()
plt.show()

total = sum(scores)
total_max = sum(maxes)
print(f'Total: {total}/{total_max}  ({total/total_max*100:.0f}%)')
if total/total_max >= 0.85:
    print('Outstanding — you are ready for advanced topics.')
elif total/total_max >= 0.65:
    print('Solid foundation — revisit any weak areas, then move forward.')
else:
    print('Keep going — review the notebooks where you feel less confident.')

---

## Congratulations — You Have Completed the 14-Week ML Course

Here is everything you covered:

| Week | Topic |
|---|---|
| 1–2 | Python, NumPy, pandas, matplotlib |
| 3–4 | Probability, distributions, hypothesis testing |
| 5–6 | Linear and logistic regression |
| 7–8 | Decision trees, random forests |
| 9–10 | Neural networks, backpropagation |
| 11–12 | Unsupervised learning, clustering, PCA |
| 13–14 | Practical Mastery — evaluation, overfitting, pipelines, capstone |

**The skills that matter most going forward:**
1. Always evaluate honestly on held-out data
2. Understand *why* a model succeeded or failed
3. Start simple, add complexity only when data demands it
4. Build things. Break things. Learn from both.